In [ ]:
#@title Settings

branch = 'master' #@param {type:"string"}
modules = 'metadata,basic_quality' #@param {type:"string"}
media_dir = 'demo_media' #@param {type:"string"}

In [ ]:
#@title Setup

%cd /content
!git clone --depth 1 -b {branch} https://github.com/seruva19/ayase.git
%cd /content/ayase

import os, sys
os.environ['USE_TF'] = '0'
os.environ['USE_JAX'] = '0'

!pip install -q pydantic-settings imageio-ffmpeg ftfy
!pip install -q --no-deps -e .
if '/content/ayase/src' not in sys.path:
  sys.path.insert(0, '/content/ayase/src')

In [ ]:
#@title Create test media

import os, cv2, numpy as np

os.makedirs(media_dir, exist_ok=True)
img = np.zeros((256, 256, 3), dtype=np.uint8)
for y in range(0, 256, 16):
  for x in range(0, 256, 16):
    img[y:y+16, x:x+16] = [240, 220, 180] if (x // 16 + y // 16) % 2 == 0 else [30, 50, 80]
cv2.imwrite(f'{media_dir}/sharp.png', img)
cv2.imwrite(f'{media_dir}/blurry.png', cv2.GaussianBlur(img, (31, 31), 10))

vw = cv2.VideoWriter(f'{media_dir}/clip.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 8.0, (256, 256))
for i in range(16):
  vw.write(np.roll(img, shift=i * 8, axis=1))
vw.release()

for name in ['sharp', 'blurry', 'clip']:
  open(f'{media_dir}/{name}.txt', 'w').write(f'{name} checkerboard\n')

In [ ]:
#@title Scan

!USE_TF=0 USE_JAX=0 ayase scan {media_dir} --modules {modules} --format json --output report.json

import pandas as pd
from ayase import AyasePipeline

pipeline = AyasePipeline(modules=modules.split(','))
pipeline.run(media_dir)
pipeline.export('report.csv', format='csv')
pd.read_csv('report.csv')